# FastSpeech 2 — English · step 2/2: training + inference (GPU)

**Settings:** Accelerator **GPU T4 ×2**, Internet **on**, **Add Input → your private prepared-data dataset** (any name: the notebook finds `English/` and `Telugu/` inside whatever is attached).

**Run:** *Save Version → Save & Run All (Commit)*. Training stops on a time budget inside Kaggle's 12 h limit.
Then open the version → **Output** and download `English_results.zip`.

Two models train at the same time, one per T4:
* **scratch** — random init, the main submission (`English/checkpoints/`)
* **finetune** — initialised from the public LJSpeech FastSpeech 2 checkpoint (ablation, `English/checkpoints/finetune/`)

In [ ]:
# ---- knobs ----
DATA_DIR        = None     # None: search everything attached under /kaggle/input (any dataset name)
TIME_BUDGET_H   = 11.3     # Kaggle hard limit is 12 h; leaves room for inference + packaging
RUN_FINETUNE    = True     # ablation run on the second GPU
FINETUNE_STEPS  = 30000
SCRATCH_STEPS   = 60000    # upper bound; the time budget may stop it first

In [ ]:
CODE_B64 = '...'
CODE_BUILD = '2026-11-01 19:11 30dcbb80'

In [ ]:
import os, sys, time, json, subprocess, shutil, glob
T0 = time.time()
PROJ = "/kaggle/working/project"
# scratch space for big intermediate files (NOT saved as notebook output)
for TMP in ("/kaggle/tmp", "/tmp/fs2"):
    try:
        os.makedirs(TMP, exist_ok=True)
        break
    except OSError:
        continue
WORK = f"{TMP}/work"
os.makedirs(WORK, exist_ok=True)

def sh(cmd, check=True):
    print("+", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return r.returncode

def hours():
    return (time.time() - T0) / 3600

# ---- unpack the project code (embedded below) ----
import base64, io, tarfile
os.makedirs(PROJ, exist_ok=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode="r:gz").extractall(PROJ)
# a stale notebook re-runs old code; this line in the log tells which build ran
print("code build:", CODE_BUILD, "| energy dtype fix:",
      ".from_numpy(energies).float()" in open(f"{PROJ}/core/utils/tools.py").read())
if not os.path.islink(f"{PROJ}/work"):
    os.symlink(WORK, f"{PROJ}/work")
os.chdir(PROJ)

# ---- assets from the reference implementation: LibriSpeech lexicon + HiFi-GAN ----
if not os.path.exists(f"{TMP}/ming024"):
    sh(f"git clone -q --depth 1 https://github.com/ming024/FastSpeech2.git {TMP}/ming024")
os.makedirs("core/lexicon", exist_ok=True)
shutil.copy(f"{TMP}/ming024/lexicon/librispeech-lexicon.txt", "core/lexicon/")
shutil.copy(f"{TMP}/ming024/hifigan/generator_universal.pth.tar.zip", "core/hifigan/")

# ---- python deps (torch, librosa, numpy come with the Kaggle image) ----
sh("pip install -q pyworld tgt g2p_en==2.1.0 soxr inflect unidecode 2>&1 | tail -2", check=False)
import nltk
for r in ["averaged_perceptron_tagger", "averaged_perceptron_tagger_eng", "cmudict"]:
    nltk.download(r, quiet=True)
import torch
print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print(f"setup done at {hours():.2f} h")

In [ ]:
def find_prepared(lang, roots, dest, tmp):
    """Locate <lang>/preprocessed/stats.json in anything attached as input, whatever the dataset
    is called and however deep it sits: an extracted folder (Kaggle unzips uploaded zips), or
    a .zip / .tar that was left packed. Puts the data at `dest`; returns where it came from."""
    import tarfile, zipfile
    def find(pattern):
        return sorted({p for r in roots for p in glob.glob(os.path.join(r, "**", pattern), recursive=True)})
    shutil.rmtree(dest, ignore_errors=True)
    key = f"{lang}/preprocessed/stats.json"
    dirs = [p[: -len("/preprocessed/stats.json")] for p in find(key)]
    if dirs:
        if len(dirs) > 1:
            print("several copies found, using the first:", *dirs, sep="\n  ")
        os.makedirs(dest)
        for sub in os.listdir(dirs[0]):   # link, read-only input is fine
            os.symlink(os.path.join(dirs[0], sub), os.path.join(dest, sub))
        return dirs[0]
    for arc in find("*.zip") + find("*.tar"):
        packed = zipfile.ZipFile(arc) if arc.endswith(".zip") else tarfile.open(arc)
        names = packed.namelist() if arc.endswith(".zip") else packed.getnames()
        hit = [n for n in names if n.endswith(key)]
        if not hit:
            continue
        prefix = hit[0][: -len("preprocessed/stats.json")]          # e.g. "fs2_prepared/Telugu/"
        unpack = os.path.join(tmp, "unpack_" + lang)
        shutil.rmtree(unpack, ignore_errors=True)
        members = [n for n in names if n.startswith(prefix)]
        if arc.endswith(".zip"):
            packed.extractall(unpack, members)
        else:
            packed.extractall(unpack, [packed.getmember(n) for n in members])
        shutil.move(os.path.join(unpack, prefix), dest)
        return arc
    print("Nothing with", key, "under", roots, "- what is attached:")
    for r in roots:
        for d, subs, files in os.walk(r):
            if d.count(os.sep) - r.count(os.sep) < 3:
                print("  ", d, f"({len(files)} files)")
    raise SystemExit(f"Prepared {lang} data not found: Add Input -> your private prepared-data dataset")

# ---- load the prepared data (attach it via Add Input first; the name does not matter) ----
src = find_prepared("English", [DATA_DIR] if DATA_DIR else ["/kaggle/input"], f"{WORK}/English", TMP)
print("prepared English data from:", src)
if os.path.exists(f"{WORK}/English/config_used.yaml"):
    shutil.copy(f"{WORK}/English/config_used.yaml", "English/config.yaml")   # same settings as the prep run
print(open("work/English/preprocessed/summary.json").read())
print(f"data ready at {hours():.2f} h")

## 4 · Training (scratch on GPU0, LJSpeech fine-tune on GPU1)

In [ ]:
def launch(cmd, log, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    f = open(log, "w")
    print(f"[GPU{gpu}] {cmd}\n   log -> {log}")
    return subprocess.Popen(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT, env=env)

def watch(procs, every_min=15):
    """procs: {name: (Popen, log)} - print the tail of each log until all finish."""
    while True:
        alive = {n: p for n, (p, l) in procs.items() if p.poll() is None}
        for n, (p, l) in procs.items():
            lines = [x for x in open(l).read().splitlines() if x.startswith(("step", "VAL", "  new best", "finished", "Traceback", "RuntimeError"))]
            print(f"--- {n} ({'running' if p.poll() is None else 'exit ' + str(p.returncode)}) @ {hours():.2f} h:", *lines[-3:], sep="\n   ")
        if not alive:
            break
        time.sleep(every_min * 60)
    for n, (p, l) in procs.items():
        if p.returncode != 0:
            print(open(l).read()[-3000:])
            raise RuntimeError(f"{n} failed")

In [ ]:
budget = TIME_BUDGET_H - hours() - 0.6
print(f"training budget: {budget:.2f} h")
procs = {}
procs["scratch"] = (launch(f"python English/train.py --total_step {SCRATCH_STEPS} --max_hours {budget:.2f}",
                           "/kaggle/working/train_scratch.log", 0), "/kaggle/working/train_scratch.log")
if RUN_FINETUNE:
    lj = None
    try:
        # ming024's Drive folder (1DOhZGlTLMbbAAFZmZGDdc77kz1PloS7F) holds LJSpeech_900000.zip next to the
        # AISHELL3 and LibriTTS checkpoints: fetch only the LJSpeech zip by its file id and unpack it
        sh("pip install -q gdown", check=False)
        pre = f"{TMP}/pretrained"
        os.makedirs(pre, exist_ok=True)
        z = f"{pre}/LJSpeech_900000.zip"
        if not os.path.exists(z):
            sh(f"gdown 1r3fYhnblBJ8hDKDSUDtidJ-BN-xAM9pe -O {z}", check=False)
        if os.path.exists(z):
            import zipfile
            zipfile.ZipFile(z).extractall(f"{pre}/LJSpeech")
        c = sorted(glob.glob(f"{pre}/LJSpeech/**/*.pth.tar", recursive=True))
        print("checkpoints in the LJSpeech zip:", c)
        lj = c[-1] if c else None
    except Exception as e:
        print("pretrained download failed:", repr(e))
    if lj:
        print("LJSpeech checkpoint:", lj)
        cfg = open("English/config.yaml").read().replace("lr_scale: 1.0", "lr_scale: 0.5").replace("warm_up_step: 4000", "warm_up_step: 1000")
        open("English/config_finetune.yaml", "w").write(cfg)
        procs["finetune"] = (launch(f"python English/train.py --config English/config_finetune.yaml --tag finetune --init_from {lj} "
                                    f"--total_step {FINETUNE_STEPS} --max_hours {budget:.2f}",
                                    "/kaggle/working/train_finetune.log", 1), "/kaggle/working/train_finetune.log")
    else:
        print("!! LJSpeech checkpoint not available - skipping the fine-tune ablation")
watch(procs)
print(f"training done at {hours():.2f} h")

## 5 · Inference on the test sentences + variance-adaptor plots

In [ ]:
import pandas as pd
sel = pd.read_csv("work/English/corpus/selection.csv")
tr = sel[sel.split == "train"]
stats = tr.groupby("speaker").agg(hours=("dur", lambda x: x.sum() / 3600), snr=("snr_db", "median"))
gcol = [c for c in sel.columns if c.lower() == "gender"]
if gcol:
    stats["gender"] = tr.groupby("speaker")[gcol[0]].first()
stats["score"] = stats.hours.rank() + stats.snr.rank()
stats = stats.sort_values("score", ascending=False)
print(stats.head(10))
spk_ids = json.load(open("work/English/preprocessed/speakers.json"))
key = lambda s: "en_" + "".join(ch for ch in str(s) if ch.isalnum())
main_spk = key(stats.index[0])
extra = []
if gcol:
    for g in stats.gender.dropna().unique():
        s = key(stats[stats.gender == g].index[0])
        if s != main_spk:
            extra.append(s)
print("main inference speaker:", main_spk, "| extra:", extra)
json.dump({"main": main_spk, "extra": extra, "table": stats.head(10).reset_index().to_dict("records")},
          open("work/English/inference_speakers.json", "w"), indent=1, default=str)

R = "Results/English"
sh(f"python English/inference.py --speaker {main_spk} --out_dir {R}")
sh(f"python English/inference.py --speaker {main_spk} --out_dir {R} --no_repair --suffix _no_repair --only E3")
for s in extra:
    sh(f"python English/inference.py --speaker {s} --out_dir {R}/other_speakers/{s}")
if os.path.exists("English/checkpoints/finetune/best.pth.tar"):
    sh(f"python English/inference.py --config English/config_finetune.yaml --ckpt English/checkpoints/finetune/best.pth.tar --speaker {main_spk} --out_dir {R}/finetune_ablation")

## 6 · Loss plots, release checkpoints, packaging

In [ ]:
runs = "scratch:work/English/logs" + (" finetune:work/English/logs/finetune" if os.path.exists("work/English/logs/finetune/losses.csv") else "")
ck = "scratch:English/checkpoints" + (" finetune:English/checkpoints/finetune" if os.path.exists("English/checkpoints/finetune/best.pth.tar") else "")
sh(f"python core/export.py --lang English --runs {runs} --ckpt_dirs {ck}")
shutil.copy("work/English/inference_speakers.json", "Results/English/logs/")
for l in glob.glob("/kaggle/working/train_*.log"):
    shutil.copy(l, "Results/English/logs/")
os.unlink(f"{PROJ}/work")      # don't save the big intermediate features
# deliverable zip: code + configs + fp16 checkpoints + results (full-precision ckpts stay in the output dir)
sh("cd /kaggle/working && zip -qr English_results.zip project/Results/English project/English "
   "-x '*/last.pth.tar' '*/best.pth.tar' '*/[0-9]*.pth.tar' && ls -lh English_results.zip")
print(f"ALL DONE in {hours():.2f} h")